
# Planet PSScene 2020 — AOI Search → Batched Orders (clip → reproject → COG) → Download

This notebook automates:
1) Reading your AOI (GeoJSON).  
2) Searching **PSScene (PlanetScope)** scenes in **2020** with a cloud filter.  
3) Placing **batched Orders** with a **toolchain**: `clip → reproject → COG`.  
4) Downloading into a tidy local folder structure you can feed into your labeling / mask-rasterization pipeline.

> **Auth:** Run `planet auth login` once so the SDK can pick it up.  
> **Quota:** Running orders consumes quota; double‑check filters before executing.


In [1]:
# %pip install -q planet geopandas shapely tqdm pandas

import os
import json
from pathlib import Path
from datetime import datetime, date

from tqdm import tqdm
import pandas as pd

from planet import Planet, data_filter, order_request, reporting

try:
    import geopandas as gpd  # optional
except Exception:
    gpd = None

AOI_DIR = Path("D:\\Thesis\\glacial_lake_detection_thesis\\Inference\\1 Getting geojson for imageries\\aoi31")   # set this to your folder path
GEOJSON_PATTERN = "*.geojson"
OUT_DIR = Path("./planet_downloads_PSScene_rgb_2025")
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR = 2025
DATE_START = f"{YEAR}-09-27"
DATE_END = f"{YEAR}-09-28"
MAX_CLOUD = 0.2

PRIMARY_BUNDLE = "visual"
#PRIMARY_BUNDLE = "analytic_8b_sr_udm2"
FALLBACK_BUNDLES = ["analytic_sr_udm2", "analytic_8b_udm2"]

REPROJECT_EPSG = "EPSG:4326"
MAX_ITEMS_PER_ORDER = 400
WAIT_MAX_ATTEMPTS = 600
ZIP_EACH_ORDER = False


In [2]:
!planet auth login
pl = Planet()
print("Planet client ready. If this fails, run `!planet auth login`.")


Logging in with authentication profile planet-user...
Opening browser to login.
Confirm the authorization code when prompted: BMBV-NSHW

Login succeeded.
Planet client ready. If this fails, run `!planet auth login`.


In [3]:
def read_aoi_geojson(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"AOI GeoJSON not found: {path}")
    obj = json.loads(path.read_text())
    if obj.get("type") == "FeatureCollection":
        geom = obj["features"][0]["geometry"]
    elif obj.get("type") == "Feature":
        geom = obj["geometry"]
    else:
        geom = obj
    if geom.get("type") not in {"Polygon", "MultiPolygon"}:
        raise ValueError("AOI must be a Polygon or MultiPolygon geometry.")
    return geom


In [4]:
def process_one_aoi(aoi_path: str|Path):
    """Run the same single-AOI flow for a given GeoJSON path.
    Creates a subfolder under OUT_DIR using the AOI filename stem.
    Returns list of created order IDs.
    """
    
    aoi_path = Path(aoi_path)
    aoi_geom = read_aoi_geojson(aoi_path)

    # Ensure sub-output dir per AOI
    aoi_name = aoi_path.stem
    aoi_out = OUT_DIR / aoi_name
    aoi_out.mkdir(parents=True, exist_ok=True)

    # ---- Search (mirrors earlier search cell) ----
    items = []
    sr_or = data_filter.or_filter([
    data_filter.asset_filter(asset_types=["ortho_visual"]),  # 8-band SR "ortho_analytic_8b_sr"
    #data_filter.asset_filter(asset_types=["ortho_analytic_sr"]),     # 4-band SR
    ])

    udm2_or = data_filter.or_filter([
        data_filter.asset_filter(asset_types=["ortho_analytic_udm2"]),
        data_filter.asset_filter(asset_types=["ortho_udm2"]),            # some items expose this
    ])
    sfilter = data_filter.and_filter([
        data_filter.std_quality_filter(),
        sr_or,
        udm2_or,
        data_filter.date_range_filter("acquired", gte=datetime.fromisoformat(DATE_START), lte=datetime.fromisoformat(DATE_END)),
        data_filter.range_filter("cloud_cover", lte=MAX_CLOUD),
    ])

    for item in pl.data.search(["PSScene"], geometry=aoi_geom, search_filter=sfilter, limit=10000):
        props = item.get("properties", {})
        items.append({
            "id": item.get("id"),
            "acquired": props.get("acquired"),
            "cloud_cover": props.get("cloud_cover"),
            "satellite_id": props.get("satellite_id"),
            "strip_id": props.get("strip_id"),
            "publishing_stage": props.get("publishing_stage"),
        })

    df = pd.DataFrame(items).sort_values(["cloud_cover", "acquired"])
    print(f"[{aoi_name}] Found {len(df)} items.")

    # Save CSV per AOI
    csv_path = aoi_out / f"search_results_{YEAR}_{aoi_name}.csv"
    df.to_csv(csv_path, index=False)
    print(f"[{aoi_name}] Wrote: {csv_path}")

    # ---- Chunk into batches ----
    def chunk(lst, n):
        for i in range(0, len(lst), n):
            yield lst[i:i+n]
    item_ids = df["id"].tolist()
    batches = list(chunk(item_ids, MAX_ITEMS_PER_ORDER))
    print(f"[{aoi_name}] Total items: {len(item_ids)} across {len(batches)} order(s).")
    return batches


In [5]:

# === Quota charge estimate (per-scene & per-order) ===
# Computes how much area (km²) Planet will deduct from your quota for this order,
# using Planet's Preferred vs Premium charging rules.
# References:
# - Preferred vs Premium minimums: 100 km² per intersecting scene vs actual area. See Orders API FAQ. 
# - New Quota Reservations API can also estimate quota for supported products.
# Docs: https://support.planet.com/hc/en-us/articles/360011216618-Orders-API-FAQ
#       https://docs.planet.com/develop/apis/quota/
from shapely.geometry import shape
from shapely.ops import transform
import pyproj
import pandas as pd
from tqdm import tqdm

PLAN_MODEL = globals().get("PLAN_MODEL", "preferred").lower()  # 'preferred' (E&R Basic) or 'premium'
EQUAL_AREA_CRS = "EPSG:6933"  # World Cylindrical Equal Area



def fetch_item_geometry(item_id: str):
    """Fetch PSScene item geometry via SDK; fallback to Data API request if needed."""
    try:
        item = pl.data.get_item("PSScene", item_id)  # SDK call (sync helper in this notebook's Planet wrapper)
        geom = item.get("geometry") or item.get("_links", {}).get("_self_geometry")  # robust-ish
        if geom:
            return shape(geom)
    except Exception as e:
        pass
    # Fallback to raw HTTP if SDK isn't available
    import os, requests, base64
    api_key = os.getenv("PL_API_KEY") or os.getenv("PL_APIKEY") or os.getenv("PL_API_KEY_ID") or os.getenv("PL_APIKEY_ID")
    if not api_key:
        # try reading the planet auth file written by `planet auth login`
        try:
            import json, pathlib
            auth_path = pathlib.Path.home()/".planet.json"
            if auth_path.exists():
                api_key = json.loads(auth_path.read_text()).get("key")
        except Exception:
            api_key = None
    if not api_key:
        raise RuntimeError("Planet API key not found. Set PL_API_KEY env var or run `!planet auth login`.")
    url = f"https://api.planet.com/data/v1/item-types/PSScene/items/{item_id}"
    resp = requests.get(url, auth=(api_key, ""))
    resp.raise_for_status()
    return shape(resp.json()["geometry"])

def estimate_for_ids(ids: list[str]) -> pd.DataFrame:
    rows = []
    for sid in tqdm(ids, desc="Fetching footprints & intersecting AOI"):
        try:
            scene_geom = fetch_item_geometry(sid)
        except Exception as e:
            # If footprint not available, assume conservative minimum charge if Preferred; else 0
            rows.append({
                "scene_id": sid,
                "intersect_km2": float('nan'),
                "charged_km2": 100.0 if PLAN_MODEL.startswith("pref") else float('nan'),
                "note": "geometry fetch failed; using minimum for Preferred" if PLAN_MODEL.startswith("pref") else "geometry fetch failed"
            })
            continue
        scene_ea = transform(to_ea, scene_geom)
        inter = aoi_ea.intersection(scene_ea)
        inter_area_km2 = inter.area / 1e6  # m² to km²

        if PLAN_MODEL.startswith("pref"):
            charged = max(100.0, inter_area_km2) if inter_area_km2 > 0 else 0.0
        else:
            charged = inter_area_km2

        rows.append({
            "scene_id": sid,
            "intersect_km2": round(inter_area_km2, 3),
            "charged_km2": round(charged, 3),
            "note": ""
        })
    df_est = pd.DataFrame(rows)
    if not df_est.empty:
        total = df_est["charged_km2"].fillna(0).sum()
        print(f"Estimated quota charge for {len(ids)} scene(s) under plan='{PLAN_MODEL}': {total:.2f} km²")
    return df_est

# Prepare transformers
to_ea = pyproj.Transformer.from_crs("EPSG:4326", EQUAL_AREA_CRS, always_xy=True).transform
aoi_files = sorted(AOI_DIR.glob(GEOJSON_PATTERN))
print(f"Batch AOIs in {AOI_DIR} → {len(aoi_files)} files")
if not aoi_files:
    raise FileNotFoundError(f"No GeoJSON files matched: {AOI_DIR / GEOJSON_PATTERN}")

all_results = {}
for i, aoi in enumerate(aoi_files, start=1):
    print(f"\n=== [{i}/{len(aoi_files)}] {aoi.name} ===")
    aoi_geom = read_aoi_geojson(aoi)
    aoi_poly = shape(aoi_geom)
    aoi_ea = transform(to_ea, aoi_poly)

    # Compute per-batch and total estimates BEFORE creating orders
    all_rows = []
    batches = process_one_aoi(aoi)
    for idx, batch_ids in enumerate(batches, start=1):
        print(f"\n=== Estimating order {idx}/{len(batches)} ({len(batch_ids)} items) ===")
        df_batch = estimate_for_ids(batch_ids)
        df_batch["order_idx"] = idx
        order_total = df_batch["charged_km2"].fillna(0).sum()
        print(f"Order {idx} subtotal (km²): {order_total:.2f}")
        all_rows.append(df_batch)

    quota_estimates = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
    quota_estimates["aoi_path"] = aoi
    display_cols = ["order_idx", "scene_id", "intersect_km2", "charged_km2", "note", "aoi_path"]
    quota_estimates = quota_estimates[display_cols] #.sort_values(["order_idx", "charged_km2"], ascending=[True, False])
    aoi_name = aoi.stem
    aoi_out = OUT_DIR / aoi_name
    csv_path = aoi_out / f"quota_{YEAR}_{aoi_name}.csv"
    quota_estimates.to_csv(csv_path, index=False)
    print(quota_estimates.head(20))


Batch AOIs in D:\Thesis\glacial_lake_detection_thesis\Inference\1 Getting geojson for imageries\aoi31 → 1 files

=== [1/1] aoi31.geojson ===
[aoi31] Found 165 items.
[aoi31] Wrote: planet_downloads_PSScene_rgb_2025\aoi31\search_results_2025_aoi31.csv
[aoi31] Total items: 165 across 1 order(s).

=== Estimating order 1/1 (165 items) ===


Fetching footprints & intersecting AOI: 100%|██████████| 165/165 [00:44<00:00,  3.73it/s]

Estimated quota charge for 165 scene(s) under plan='preferred': 63675.21 km²
Order 1 subtotal (km²): 63675.21
    order_idx                 scene_id  intersect_km2  charged_km2 note  \
0           1  20250927_060343_66_2523          0.030      100.000        
1           1  20250927_060345_84_2523          0.016      100.000        
2           1  20250927_060348_02_2523          0.093      100.000        
3           1  20250927_060350_20_2523          0.269      100.000        
4           1  20250927_060352_38_2523          0.038      100.000        
5           1  20250927_061113_79_24db        520.239      520.239        
6           1  20250927_061115_96_24db        580.260      580.260        
7           1  20250927_061118_12_24db        589.691      589.691        
8           1  20250927_061120_29_24db        530.684      530.684        
9           1  20250927_061122_46_24db        527.376      527.376        
10          1  20250927_061124_62_24db        502.713      502.71

In [5]:
aoi_files = sorted(AOI_DIR.glob(GEOJSON_PATTERN))
path_list = []
for aoi_file in aoi_files:
    aoi_name = aoi_file.stem
    aoi_out = OUT_DIR / aoi_name
    csv_path = aoi_out / f"quota_{YEAR}_{aoi_name}.csv"
    path_list.append(csv_path)
frames = [pd.read_csv(p, low_memory=False).assign(source_file=p.name) for p in path_list]

# concatenate into a single DataFrame
df = pd.concat(frames, ignore_index=True, sort=False)
df.to_csv(OUT_DIR / "quota_concatenated.csv", index=False)

In [6]:
df = pd.read_csv(OUT_DIR / "quota_concatenated.csv")
order_df = df.tail(54)
order_df


,order_idx,scene_id,intersect_km2,charged_km2,note,aoi_path,source_file
54,1,20250927_062230_80_254a,685.913,685.913,NaN,D:\Thesis\glacial_lake_detection_thesis\Infere...,quota_2025_aoi31.csv
55,1,20250927_062233_11_254a,293.207,293.207,NaN,D:\Thesis\glacial_lake_detection_thesis\Infere...,quota_2025_aoi31.csv
56,1,20250927_062535_79_2539,362.477,362.477,NaN,D:\Thesis\glacial_lake_detection_thesis\Infere...,quota_2025_aoi31.csv
57,1,20250927_062542_72_2539,461.110,461.110,NaN,D:\Thesis\glacial_lake_detection_thesis\Infere...,quota_2025_aoi31.csv
58,1,20250927_062545_03_2539,486.526,486.526,NaN,D:\Thesis\glacial_lake_detection_thesis\Infere...,quota_2025_aoi31.csv
59,1,20250927_062547_34_2539,427.939,427.939,NaN,D:\Thesis\glacial_lake_detection_thesis\Infere...,quota_2025_aoi31.csv
60,1,20250927_062549_65_2539,449.122,449.122,NaN,D:\Thesis\glacial_lake_detection_thesis\Infere...,quota_2025_aoi31.csv
61,1,20250927_062551_96_2539,596.497,596.497,NaN,D:\Thesis\glacial_lake_detection_thesis\Infere...,quota_2025_aoi31.csv
62,1,20250927_062554_27_2539,681.906,681.906,NaN,D:\Thesis\glacial_lake_detection_thesis\Infere...,quota_2025_aoi31.csv
63,1,20250927_062556_58_2539,569.491,569.491,NaN,D:\Thesis\glacial_lake_detection_thesis\Infere...,quota_2025_aoi31.csv


In [7]:
for idx, row in order_df.iterrows():
    print(row["scene_id"])


20250927_062230_80_254a
20250927_062233_11_254a
20250927_062535_79_2539
20250927_062542_72_2539
20250927_062545_03_2539
20250927_062547_34_2539
20250927_062549_65_2539
20250927_062551_96_2539
20250927_062554_27_2539
20250927_062556_58_2539
20250927_062558_89_2539
20250927_062628_17_2531
20250927_062636_47_2531
20250927_062638_55_2531
20250927_062640_62_2531
20250927_062642_70_2531
20250927_062644_77_2531
20250927_062646_84_2531
20250927_062648_92_2531
20250927_061737_42_253c
20250927_061739_71_253c
20250927_061742_00_253c
20250927_062201_04_24ae
20250927_062210_05_254a
20250927_062212_35_254a
20250927_062221_58_254a
20250927_062538_10_2539
20250927_062540_41_2539
20250927_062205_43_254a
20250927_062216_97_254a
20250927_062219_27_254a
20250927_062626_10_2531
20250927_061214_37_2511
20250927_061857_41_24fe
20250927_062159_11_24ae
20250927_061102_97_24db
20250927_061111_63_24db
20250927_061857_44_2547
20250927_061107_30_24db
20250927_061216_54_2511
20250927_061836_87_24fe
20250927_061231_

In [ ]:
for idx, row in order_df.iterrows():
    aoi_geom = read_aoi_geojson(Path(row["aoi_path"]))

    tools = [
        order_request.clip_tool(aoi_geom),
        order_request.reproject_tool(REPROJECT_EPSG),
        order_request.file_format_tool("COG"),
    ]

    delivery_cfg = None
    if ZIP_EACH_ORDER:
        delivery_cfg = order_request.delivery(archive_type="zip", single_archive=True)

    order_ids = []
    
    order_name = f"PSScene_{YEAR}_AOI_{idx:02d}_delta"
    print(f"Creating order with scene ID {row["scene_id"]}")

    req = order_request.build_request(
        name=order_name,
        products=[order_request.product(item_ids=[row["scene_id"]], product_bundle=PRIMARY_BUNDLE, item_type="PSScene",
                                            fallback_bundle=FALLBACK_BUNDLES or None)],
        tools=tools,
        delivery=delivery_cfg
    )
    order = pl.orders.create_order(req)
    oid = order["id"]
    order_ids.append(oid)
    print("  created:", oid)

    with reporting.StateBar(state="creating") as bar:
        pl.orders.wait(oid, callback=bar.update_state, max_attempts=WAIT_MAX_ATTEMPTS)

    order_dir = OUT_DIR / f"order_{idx:02d}_{oid}"
    order_dir.mkdir(parents=True, exist_ok=True)
    print("  downloading to:", order_dir)
    pl.orders.download_order(oid, directory=str(order_dir), overwrite=True)

    order_ids


Creating order with scene ID 20250927_062230_80_254a
